# GP09 — Spatial Join: Ενδεικτική λύση

Το spatial join συνδέει features με βάση τη χωρική τους σχέση, όχι κοινό key. Τα services είναι left και τα districts right: κρατάμε τη Point geometry των services και μεταφέρουμε district attributes. Το `left` χρησιμοποιείται για QA, ενώ το `inner` για matched analysis.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
root = Path('data/gp09')
df = pd.read_csv(root / 'services.csv')
services = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326')
districts = gpd.read_file(root / 'districts.geojson')
assert services.crs == districts.crs


In [ ]:
joined_left = gpd.sjoin(services, districts, how='left', predicate='within', lsuffix='service', rsuffix='district')
joined_inner = gpd.sjoin(services, districts, how='inner', predicate='within', lsuffix='service', rsuffix='district')
print('services:', len(services), 'left:', len(joined_left), 'inner:', len(joined_inner))
print(joined_left[['service_id','district_name']])


In [ ]:
counts = (joined_inner.groupby(['district_id','district_name'], dropna=False).size().reset_index(name='service_count'))
print(counts)
unmatched = joined_left['district_id'].isna().sum()
print('unmatched:', unmatched)


In [ ]:
assert len(joined_left) >= len(services)
assert len(joined_inner) == 5
assert unmatched == 1
assert joined_left.geometry.geom_type.eq('Point').all()
assert joined_left.crs.to_epsg() == 4326
assert counts['service_count'].sum() == len(joined_inner)
assert joined_left.total_bounds[0] >= services.total_bounds[0]


In [ ]:
joined_left.to_file('gp09_services_by_district.gpkg', layer='services', driver='GPKG')
counts.to_csv('gp09_service_counts.csv', index=False)
print('exported join and aggregation')
